# Grenoble: historical cycling 2017–2022 + current 2026 cycling and car roads
Uses the same classifier and simplifier as before. The 2026 network is a NEW download, so server availability matters. The 2026 builder caches the cycling and car graphs separately, and its timeout is 120 seconds rather than 600. If a request times out, rerun only its 2026 cell; previous checkpoint files remain.


# Grenoble: 2017–2022 cycling LTS 1–4; 2026 cycling + car LTS 1–5

Minimal update to your previous workflow. **2017–2022** use only saved historical cycling networks. **2026** uses the currently available OpenStreetMap bike + driving networks, joined **before** simplification. All metrics are computed **after** simplification, with LTS 5 excluded from length and LCC. No Matplotlib; direct Jupyter Plotly topology plots.

Run from the folder containing these Python files. This notebook does not change your previous output PKLs.

In [3]:
from pathlib import Path
import pickle
import networkx as nx
from IPython.display import display
from grenoble_metrics_repair import ROOT, build_saved_metrics
from grenoble_simplification_exact import check_graph, plot_topology
from grenoble_metrics_exponential import annual_metrics, exponential_weights
from grenoble_2026_extension import build_2026_combined
print('Output folder:', ROOT)

Output folder: grenoble_lts_output\historical


## 1. Read/simplify cycling-only graphs for 2017–2022
Existing saved historical checkpoints are reused. **No Overpass calls** in this section. If a year is missing, generate it using your already-working historical notebook first.

In [5]:
from pathlib import Path
from shapely import wkb
import networkx as nx
from grenoble_metrics_repair import ROOT, build_saved_metrics
from grenoble_simplification_exact import check_graph
from grenoble_workflow_fixed import prepare_bike, save_pickle
from grenoble_lts_historical import build_historical_lts_graph
import grenoble_lts_graph as original

# The polygon is needed here, because this cell may download data
with open('grenoble_polygon.wkb', 'rb') as f:
    poly = wkb.loads(f.read())

def ensure_bike_year(year, timeout=600, overpass_url="https://overpass-api.de/api"):
    folder = Path(ROOT) / str(year)
    prepared = folder / f'grenoble_lts_{year}_bike_prepared.pkl'
    simplified = folder / f'grenoble_lts_{year}_simplified.pkl'
    if prepared.exists() or simplified.exists():
        return
    print(f'[{year}] no saved graph, downloading historical OSM snapshot...', flush=True)
    old_plot = original.plot_lts_graph
    original.plot_lts_graph = lambda *a, **k: None   # no Matplotlib
    try:
        _, main, _, _ = build_historical_lts_graph(
            poly=poly, snapshot_date=f'{year}-12-31T23:59:59Z',
            output_dir=folder, requests_timeout=timeout,
            overpass_url=overpass_url, cache_root='cache_historical_osm')
    finally:
        original.plot_lts_graph = old_plot
    bike = prepare_bike(main); bike.graph['year'] = year
    save_pickle(bike, prepared)
    print(f'[{year}] saved {prepared}', flush=True)

metric_graphs = {}
for year in range(2017, 2023):
    ensure_bike_year(year)
    G = build_saved_metrics(year)
    diagnostics = check_graph(G)
    assert type(G) is nx.Graph and diagnostics['connected']
    assert diagnostics['remaining_contractible_degree2_nodes'] == 0
    assert all(1 <= int(d['key']) <= 4 for _, _, d in G.edges(data=True))
    metric_graphs[year] = G

[2017] no saved graph, downloading historical OSM snapshot...
[2017] saved grenoble_lts_output\historical\2017\grenoble_lts_2017_bike_prepared.pkl
2017: from grenoble_lts_2017_bike_prepared.pkl; 4663 -> 4501 nodes; 6687 -> 6469 edges; contracted=162
{'nodes': 4501, 'edges': 6469, 'multigraph': False, 'connected': True, 'remaining_contractible_degree2_nodes': 0, 'lts_counts': {2: 3380, 3: 676, 1: 2363, 4: 50}, 'mandatory_lts5_edges': 0}
[2018] no saved graph, downloading historical OSM snapshot...
[2018] saved grenoble_lts_output\historical\2018\grenoble_lts_2018_bike_prepared.pkl
2018: from grenoble_lts_2018_bike_prepared.pkl; 4768 -> 4593 nodes; 6804 -> 6569 edges; contracted=175
{'nodes': 4593, 'edges': 6569, 'multigraph': False, 'connected': True, 'remaining_contractible_degree2_nodes': 0, 'lts_counts': {2: 3388, 3: 670, 1: 2461, 4: 50}, 'mandatory_lts5_edges': 0}
[2019] no saved graph, downloading historical OSM snapshot...
[2019] saved grenoble_lts_output\historical\2019\grenoble_

KeyboardInterrupt: 

In [4]:
# Run only if these packages are missing from your Bikes_clean kernel:
# %pip install osmnx networkx shapely plotly nbformat pandas
from pathlib import Path
from shapely import wkb
import pandas as pd
from grenoble_lts_2017_2022_mixed import (
    YEARS, graph_path, bike_checkpoint_path, get_year, build_all,
    load_year, edge_summary, plot_year,
)
with open('grenoble_polygon.wkb', 'rb') as f:
    poly = wkb.loads(f.read())
print('Polygon valid:', poly.is_valid, '| Type:', poly.geom_type)

Polygon valid: True | Type: Polygon


## 2. Build/load combined **2026** graph and simplify both layers together

For 2026, download the **current** OSM bicycle and car road networks (not a future December snapshot). The first run needs internet/Overpass; future runs reuse the simplified `pkl`, or the saved combined checkpoint. Add **your existing `poly`** from the Grenoble polygon cell when downloading the first time. Historical 2017–2022 files are unchanged.

In [5]:
# Keep the same `poly` you already load in your Grenoble notebook.
# To download 2026 for the first time, first execute your existing polygon-loading cell.
G_opt_2026 = build_2026_combined(
    poly=globals().get('poly'),
    rebuild=False,
    overpass_url="https://overpass.private.coffee/api",
    timeout=120,
)
check = check_graph(G_opt_2026)
assert type(G_opt_2026) is nx.Graph and check['connected']
assert check['remaining_contractible_degree2_nodes'] == 0
metric_graphs[2026] = G_opt_2026   # metrics calculated from the combined SIMPLIFIED graph
print('2026 simplified graph and metric input are the same object.')

[2026] Overpass endpoint: https://overpass.private.coffee/api
[2026] HTTP timeout: 120 seconds
[2026] Downloading CURRENT bicycle network...


ReadTimeout: HTTPSConnectionPool(host='overpass.private.coffee', port=443): Read timed out. (read timeout=120)

## 3. Exponential LCC and normalized network length (simplified graphs only)

The weights are `exp[-(γ-1)]` normalized across γ = 1,2,3,4 (κ=1). The denominator is fixed across **the seven analyzed years** (default: maximum annual cycling length). LTS 5 adds graph topology but contributes **zero** to cycling length and threshold LCC.

In [ ]:
KAPPA = 1.0
REFERENCE_LENGTH_KM = None  # default max cycling length across 2017–2022 & 2026
metrics = annual_metrics(metric_graphs,kappa=KAPPA,reference_length_km=REFERENCE_LENGTH_KM)
cols = ['year','cycling_length_km','normalized_network_length','lcc_lts1_km','lcc_lts2_km','lcc_lts3_km','lcc_lts4_km','weighted_lcc_normalized','lts5_edges','lts5_mandatory_edges']
display(metrics[cols].round(5))
print('Weights:',exponential_weights(KAPPA))

output = Path(ROOT)
output.mkdir(parents=True,exist_ok=True)
txt_path=output/'grenoble_2017_2022_and_2026_SIMPLIFIED_exponential_metrics.txt'
csv_path=output/'grenoble_2017_2022_and_2026_SIMPLIFIED_exponential_metrics.csv'
with txt_path.open('w', encoding='utf-8') as f:
    f.write('Grenoble simplified cycling infrastructure metrics, 2017–2022 and 2026\n')
    f.write('2017–2022: cycling-only LTS1-4 simplified graphs\n')
    f.write('2026: cycling + missing-car LTS1-5 graph simplified together; LTS5 excluded only from metrics\n')
    f.write('LCC: largest component by node count at each LTS threshold, length = sum of its edges\n')
    f.write('Exponential threshold weights; kappa=1; no OD demand\n')
    f.write(f'Reference cycling length (km): {metrics.reference_length_km.iloc[0]:.6f}\n\n')
    f.write(metrics.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
    f.write('\n')
metrics.to_csv(csv_path,index=False)
print('Saved:',txt_path,'and',csv_path)

## 4. Plot all simplified topologies **inside Jupyter**
The lines connect the remaining **graph nodes** rather than reconstructing every old street-geometry vertex. Black dots are retained junction nodes. This is a topology/simplification check, not a detailed road map.

In [ ]:
for year, G in sorted(metric_graphs.items()):
    d=check_graph(G)
    length_km=sum(float(e['distance']) for _,_,e in G.edges(data=True) if int(e['key'])<=4)/1000
    recorded=metrics.loc[metrics.year==year,'cycling_length_km'].iloc[0]
    assert abs(length_km-recorded)<1e-7
    print(f'{year}: {G.number_of_nodes():,} nodes; {G.number_of_edges():,} edges; cycling length {length_km:.3f} km')
    plot_topology(G,show_nodes=True)

## 5. Reload the simplified 2026 optimization graph
All LTS1–5 and mandatory-upgrade attributes are saved in this graph. No new network downloads required.

In [ ]:
path_2026 = Path(ROOT)/'2026'/'grenoble_lts5_2026_SIMPLE_optimization.pkl'
with path_2026.open('rb') as f:
    G_2026_saved=pickle.load(f)  # only open your own trusted pickle files
print(type(G_2026_saved), G_2026_saved.number_of_nodes(), G_2026_saved.number_of_edges())
print('Mandatory LTS5 edges:',sum(d['key']==5 and bool(d.get('mandatory_upgrade_to_lts1')) for _,_,d in G_2026_saved.edges(data=True)))